In [ ]:
import os
import glob
import re

def procesar_archivo(ruta_txt):
    # Genera el nombre del archivo de salida manteniendo el mismo nombre base pero con extensión .dat
    nombre_base = os.path.splitext(ruta_txt)[0]
    ruta_dat = f"{nombre_base}.dat"

    lineas_modificadas = []

    with open(ruta_txt, 'r', encoding='utf-8') as f:
        for linea in f:
            # Omitir líneas vacías
            if not linea.strip():
                continue

            # Separar la línea por cualquier espacio o tabulación en blanco
            columnas = linea.strip().split()

            if not columnas:
                continue

            # Limpiar la primera columna: extraer solo dígitos y dos puntos (ej: "C00:00:18" -> "00:00:18")
            # Si hay varias letras al inicio o caracteres extra, esto se queda estrictamente con el formato de tiempo.
            tiempo_original = columnas[0]

            # Usamos expresión regular para buscar el patrón de tiempo (números y dos puntos)
            coincidencia = re.search(r'(\d{1,2}:\d{2}:\d{2})', tiempo_original)

            if coincidencia:
                columnas[0] = coincidencia.group(1)
            else:
                # Fallback por si el formato varía un poco: remueve todas las letras al inicio
                tiempo_limpio = re.sub(r'^[a-zA-Z]+', '', tiempo_original)
                columnas[0] = tiempo_limpio

            # Unir las columnas usando un espacio estándar (ideal para Gnuplot en Linux)
            linea_limpia = " ".join(columnas)
            lineas_modificadas.append(linea_limpia)

    # Guardar el nuevo archivo con formato .dat
    with open(ruta_dat, 'w', encoding='utf-8') as f:
        f.write("\n".join(lineas_modificadas) + "\n")

    print(f"[OK] Convertido: '{ruta_txt}'  --->  '{ruta_dat}'")

def main():
    # Buscar todos los archivos .txt en el directorio actual
    archivos_txt = glob.glob("*.txt")

    if not archivos_txt:
        print("No se encontraron archivos .txt en el directorio actual.")
        return

    print(f"Se encontraron {len(archivos_txt)} archivo(s) .txt para procesar.\n")

    for archivo in archivos_txt:
        procesar_archivo(archivo)

    print("\n¡Proceso finalizado con éxito!")

if __name__ == "__main__":
    main()

Se encontraron 7 archivo(s) .txt para procesar.

[OK] Convertido: '2026-08-12.txt'  --->  '2026-08-12.dat'
[OK] Convertido: '2026-08-14.txt'  --->  '2026-08-14.dat'
[OK] Convertido: '2026-08-10.txt'  --->  '2026-08-10.dat'
[OK] Convertido: '2026-08-09.txt'  --->  '2026-08-09.dat'
[OK] Convertido: '2026-08-13.txt'  --->  '2026-08-13.dat'
[OK] Convertido: '2026-08-16.txt'  --->  '2026-08-16.dat'
[OK] Convertido: '2026-08-15.txt'  --->  '2026-08-15.dat'

¡Proceso finalizado con éxito!


In [ ]:
import os
import glob
import re
import pandas as pd

def limpiar_texto_tiempo(valor):
    """Limpia cualquier texto o letras al inicio de la celda de tiempo."""
    if pd.isna(valor):
        return ""

    valor_str = str(valor).strip()

    # Buscar patrón estándar de tiempo (HH:MM:SS)
    match = re.search(r'(\d{1,2}:\d{2}:\d{2})', valor_str)
    if match:
        return match.group(1)
    else:
        # Si no encaja exactamente, remueve cualquier letra al inicio
        return re.sub(r'^[a-zA-Z]+', '', valor_str)

def procesar_excel(ruta_xlsx):
    # Genera el nombre de salida cambiando la extensión a .dat
    nombre_base = os.path.splitext(ruta_xlsx)[0]
    ruta_dat = f"{nombre_base}.dat"

    try:
        # Leer el archivo Excel (lee la primera hoja por defecto)
        # header=None asegura que no asuma la primera fila como encabezado si tus datos empiezan directo
        df = pd.read_excel(ruta_xlsx, header=None)

        if df.empty:
            print(f"[AVISO] El archivo '{ruta_xlsx}' está vacío.")
            return

        # Aplicar la limpieza estrictamente a la primera columna (índice 0)
        df.iloc[:, 0] = df.iloc[:, 0].apply(limpiar_texto_tiempo)

        # Exportar a un archivo de texto plano separado por espacios (formato ideal para Gnuplot)
        # index=False y header=False evitan que se guarden los índices de las filas o nombres de columnas de pandas
        df.to_csv(ruta_dat, sep=' ', index=False, header=False, na_rep='')

        print(f"[OK] Convertido Excel: '{ruta_xlsx}'  --->  '{ruta_dat}'")

    except Exception as e:
        print(f"[ERROR] No se pudo procesar '{ruta_xlsx}': {e}")

def main():
    # Buscar todos los archivos .xlsx en el directorio actual
    archivos_xlsx = glob.glob("*.xlsx")

    if not archivos_xlsx:
        print("No se encontraron archivos .xlsx en el directorio actual.")
        return

    print(f"Se encontraron {len(archivos_xlsx)} archivo(s) .xlsx para procesar.\n")

    for archivo in archivos_xlsx:
        procesar_excel(archivo)

    print("\n¡Proceso de archivos Excel finalizado con éxito!")

if __name__ == "__main__":
    main()

Se encontraron 14 archivo(s) .xlsx para procesar.

[OK] Convertido Excel: '2026-07-16.xlsx'  --->  '2026-07-16.dat'
[OK] Convertido Excel: '2026-07-27.xlsx'  --->  '2026-07-27.dat'
[OK] Convertido Excel: '2026-07-28.xlsx'  --->  '2026-07-28.dat'
[OK] Convertido Excel: '2026-07-15.xlsx'  --->  '2026-07-15.dat'
[OK] Convertido Excel: '2026-07-23.xlsx'  --->  '2026-07-23.dat'
[OK] Convertido Excel: '2026-07-17.xlsx'  --->  '2026-07-17.dat'
[OK] Convertido Excel: '2026-08-08.xlsx'  --->  '2026-08-08.dat'
[OK] Convertido Excel: '2026-07-18.xlsx'  --->  '2026-07-18.dat'
[OK] Convertido Excel: '2026-07-11.xlsx'  --->  '2026-07-11.dat'
[OK] Convertido Excel: '2026-07-19.xlsx'  --->  '2026-07-19.dat'
[OK] Convertido Excel: '2026-07-26.xlsx'  --->  '2026-07-26.dat'
[OK] Convertido Excel: '2026-07-25.xlsx'  --->  '2026-07-25.dat'
[OK] Convertido Excel: '2026-07-22.xlsx'  --->  '2026-07-22.dat'
[OK] Convertido Excel: '2026-07-24.xlsx'  --->  '2026-07-24.dat'

¡Proceso de archivos Excel finalizado 